# 03 — MLP and XGBoost baselines

**No graphs in this notebook.** The numerical/categorical arrays must have been fitted only on the *training* chronology in `scripts/build_dataset.py`. The MLP and GraphSAGE use the same `NodeEncoder` architecture; XGBoost uses the same processed columns with `feature_types=(q,c)` and a native categorical tree split.

The complete reference implementations live in `scripts/train_mlp.py` and `scripts/train_xgboost.py`, not duplicated in this notebook. This notebook checks data and optionally runs them from a separate process.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository:", ROOT)

In [ ]:
from fraud_graph.data import load_processed
from fraud_graph.paths import PROCESSED_DIR
arr, meta = load_processed()
print("Numeric shape:", arr["numeric"].shape)
print("Categorical shape:", arr["categorical"].shape)
print("Fraud rate by split:")
for split in ("train", "val", "test"):
    mask = arr[f"{split}_mask"]
    print(split, mask.sum(), arr["y"][mask].mean())

In [ ]:
from fraud_graph.models import embedding_dimensions
embedding_dims = embedding_dimensions(meta["category_sizes"])
print("Categorical fields:", len(meta["categorical_cols"]))
print("Total embedding dimension:", sum(embedding_dims.values()))

## Train model entry points

Setting `RUN_TRAINING=True` runs full experiments that can take many minutes. The scripts restore validation-best checkpoints and evaluate the test set only once at the end. Original results are recorded separately in `docs/reference_results.csv`.

In [ ]:
import subprocess
RUN_TRAINING = False
if RUN_TRAINING:
    subprocess.run(["uv", "run", "python", "scripts/train_mlp.py", "--seed", "42"], cwd=ROOT, check=True)
    subprocess.run(["uv", "run", "python", "scripts/train_xgboost.py", "--seed", "42"], cwd=ROOT, check=True)
else:
    print("Set RUN_TRAINING=True when you are ready to rerun the baselines.")